# Parameters

In [6]:
import os
import numpy as np

# set n and k
n = int(1e3)
k = int(1e1)

# set folder name based on n
n_readable = f"{n:.0e}".replace(".0", "").replace("+0", "")
experiment_dim = "dim_" + n_readable
load_folder = "./private/data/"+experiment_dim+"/"

# check if load_folder exists
if not os.path.exists(load_folder):
    raise FileNotFoundError(f"Folder {load_folder} does not exist. Please run generate_matrices.ipynb first to generate the matrices.")



# check if n/k has no remainder
if n % k == 0:
    # if it has no remainder, we can simply divide n by k
    m = n // k
    sizes = [n // k] * k
else: 
    raise ValueError("n must be divisible by k")


print(f"n: {n}, k: {k}, sizes: {sizes}")

n: 1000, k: 10, sizes: [100, 100, 100, 100, 100, 100, 100, 100, 100, 100]


# Generate x_u for q

### Scenario 1: solution inside boundaries

In [7]:
x_u_sc1 = np.zeros(n)
start = 0

for i, size in enumerate(sizes):
    # Genera valori strettamente positivi che sommano a 1 per il blocco corrente
    blocco = np.random.dirichlet(np.ones(size))
    x_u_sc1[start:start+size] = blocco
    
    print(f"Blocco {i+1} ({size} var): {np.round(blocco, 3)} | Somma: {np.sum(blocco):.2f}")
    start += size

print(f"\nx_u_sc1 finale: {np.round(x_u_sc1, 3)}")

Blocco 1 (100 var): [0.008 0.012 0.007 0.001 0.006 0.    0.004 0.021 0.026 0.007 0.002 0.005
 0.023 0.011 0.004 0.009 0.001 0.014 0.013 0.003 0.017 0.007 0.001 0.016
 0.01  0.021 0.003 0.038 0.002 0.002 0.001 0.004 0.005 0.004 0.    0.023
 0.012 0.022 0.002 0.002 0.004 0.003 0.009 0.01  0.019 0.03  0.003 0.025
 0.002 0.007 0.016 0.003 0.06  0.002 0.014 0.034 0.001 0.014 0.012 0.004
 0.001 0.022 0.003 0.032 0.02  0.002 0.024 0.005 0.001 0.015 0.009 0.008
 0.015 0.012 0.001 0.023 0.001 0.017 0.001 0.001 0.014 0.003 0.014 0.011
 0.013 0.006 0.005 0.013 0.001 0.005 0.005 0.004 0.002 0.    0.007 0.006
 0.002 0.003 0.027 0.004] | Somma: 1.00
Blocco 2 (100 var): [0.001 0.004 0.02  0.019 0.017 0.018 0.005 0.    0.002 0.001 0.009 0.012
 0.01  0.002 0.014 0.014 0.035 0.032 0.018 0.009 0.019 0.002 0.004 0.019
 0.002 0.016 0.009 0.002 0.008 0.003 0.006 0.003 0.01  0.008 0.008 0.007
 0.008 0.001 0.002 0.001 0.019 0.02  0.011 0.012 0.002 0.014 0.008 0.002
 0.001 0.006 0.01  0.024 0.031 0.005 0.026 0

### Scenario 2: solution very outside boundaries

In [8]:
x_u_sc2 = np.zeros(n)
start = 0

for i, size in enumerate(sizes):
    if size == 1:
        x_u_sc2[start] = 1.0 # Se un blocco ha 1 sola variabile, deve per forza essere 1
    else:
        # Genera numeri con varianza estrema (es. tra -20 e +20)
        blocco = np.random.uniform(-20, 20, size)
        
        # Correzione: calcola quanto la somma differisce da 1 e trasla tutti i valori
        errore = np.sum(blocco) - 1.0
        blocco = blocco - (errore / size)
        
        x_u_sc2[start:start+size] = blocco
    
    print(f"Blocco {i+1} ({size} var): {np.round(x_u_sc2[start:start+size], 3)} | Somma: {np.sum(x_u_sc2[start:start+size]):.2f}")
    start += size

print(f"\nx_u_sc2 finale: {np.round(x_u_sc2, 3)}")

Blocco 1 (100 var): [ -1.786   3.149   2.363  -9.296   1.082  -6.214  -7.019   2.016  12.358
  15.067   5.033  14.887 -15.494  20.441  -3.463   4.79   12.518  11.89
  11.514   5.759   0.25   11.103 -17.495  -5.944   7.561  11.078   0.425
 -14.358 -17.026  -5.165 -13.857   1.497  -6.432   7.956 -18.744   4.061
  -3.337 -12.773 -14.363   4.977   5.835   9.223  11.502 -17.937  -7.397
  19.101 -18.085  -9.807  11.449 -19.173 -13.281  10.06    0.557  18.165
 -19.365  -3.44  -12.15    2.234   6.64  -18.1   -18.098  -4.884 -14.702
   5.491  -0.907  -5.101   1.982  17.843   0.405  -1.932  -5.735   5.338
   2.04  -18.539  -0.713  20.215   2.647   2.481   4.02   14.323   2.465
   6.213  -5.243   7.139   9.932  17.119 -11.942  17.51   -3.398 -17.3
  18.912  13.312  16.575  -9.588  13.346  -8.602 -19.03    5.288 -17.561
   4.641] | Somma: 1.00
Blocco 2 (100 var): [  4.797  -1.187   8.848   6.488   6.403   6.765  19.252  -0.556 -14.107
   7.362 -12.547 -14.19  -12.15  -16.422   1.324 -17.736   1.81

### Scenario 3: solution just a bit outside boundaries

In [9]:
x_u_sc3 = np.zeros(n)
start = 0

for i, size in enumerate(sizes):
    blocco = np.zeros(size)
    if size == 1:
        blocco[0] = 1.0 # Non si può violare se c'è 1 sola variabile
    else:
        # 1. Scegli a caso UNA SOLA variabile da rendere leggermente negativa
        idx_neg = np.random.randint(size)
        valore_negativo = -np.random.uniform(0.01, 0.1) # es. -0.05
        blocco[idx_neg] = valore_negativo
        
        # 2. La somma degli altri deve compensare per arrivare a 1
        somma_rimanente = 1.0 - valore_negativo
        
        # 3. Distribuisci la parte positiva rimanente tra gli altri elementi
        indici_restanti = [j for j in range(size) if j != idx_neg]
        blocco[indici_restanti] = np.random.dirichlet(np.ones(size - 1)) * somma_rimanente

    x_u_sc3[start:start+size] = blocco
    print(f"Blocco {i+1} ({size} var): {np.round(x_u_sc3[start:start+size], 3)} | Somma: {np.sum(x_u_sc3[start:start+size]):.2f}")
    start += size

print(f"\nx_u_sc3 finale: {np.round(x_u_sc3, 3)}")

Blocco 1 (100 var): [ 0.002  0.016  0.004 -0.047  0.006  0.004  0.022  0.015  0.008  0.01
  0.001  0.004  0.013  0.009  0.006  0.006  0.004  0.001  0.007  0.001
  0.002  0.008  0.005  0.015  0.001  0.015  0.004  0.008  0.022  0.035
  0.001  0.01   0.005  0.018  0.009  0.003  0.007  0.014  0.     0.002
  0.025  0.027  0.006  0.013  0.003  0.011  0.003  0.008  0.016  0.003
  0.002  0.02   0.003  0.031  0.006  0.014  0.004  0.016  0.007  0.012
  0.015  0.02   0.026  0.005  0.023  0.013  0.009  0.034  0.004  0.013
  0.007  0.013  0.01   0.011  0.01   0.01   0.01   0.016  0.01   0.007
  0.004  0.014  0.     0.011  0.01   0.009  0.002  0.007  0.059  0.001
  0.028  0.017  0.002  0.011  0.035  0.008  0.     0.007  0.003  0.004] | Somma: 1.00
Blocco 2 (100 var): [ 0.021  0.005  0.017  0.     0.013  0.007  0.006  0.     0.005  0.027
  0.006  0.001  0.002  0.029  0.002  0.001  0.     0.001  0.001  0.012
  0.037  0.001  0.02   0.001  0.021  0.009  0.008  0.007  0.001  0.006
  0.014  0.035  0.002  

# Q_well

In [10]:
# load Q_well from npz file
Q_well = np.load(load_folder+"matrices.npz")["Q_well"]

q_well_sc1 = Q_well @ x_u_sc1
q_well_sc2 = Q_well @ x_u_sc2
q_well_sc3 = Q_well @ x_u_sc3

# Q_ill

In [11]:
# load Q_ill from npz file
Q_ill = np.load(load_folder+"matrices.npz")["Q_ill"]

q_ill_sc1 = Q_ill @ x_u_sc1
q_ill_sc2 = Q_ill @ x_u_sc2
q_ill_sc3 = Q_ill @ x_u_sc3

# Save generated data

In [12]:
# save q_well_sc1, q_well_sc2, q_well_sc3, q_ill_sc1, q_ill_sc2, q_ill_sc3 in npz file
np.savez(load_folder+"vectors.npz",
         q_well_sc1=q_well_sc1, q_well_sc2=q_well_sc2, q_well_sc3=q_well_sc3,
         q_ill_sc1=q_ill_sc1, q_ill_sc2=q_ill_sc2, q_ill_sc3=q_ill_sc3)